# 04 - Train MLP
Đánh giá MLP PyTorch thật; log theo từng fold được ghi vào file log.


## Chuẩn bị


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT = PROJECT_ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)



## Hàm: `log`


In [ ]:
def log(message):
    print(message)
    with (OUTPUT / "train_log.txt").open("a", encoding="utf-8") as file:
        file.write(f"{message}\n")

import pandas as pd
from IPython import get_ipython
get_ipython().run_line_magic("run", str(PROJECT_ROOT / "house_price_pipeline.ipynb"))
from IPython import get_ipython
get_ipython().run_line_magic("run", str(PROJECT_ROOT / "pytorch_mlp.ipynb"))



## Hàm: `mlp_cv`


In [ ]:
def mlp_cv(train: pd.DataFrame, folds: int, random_state: int, quick: bool):
    # Cross-validation riêng cho MLP PyTorch nhưng vẫn dùng tiền xử lý của sklearn.
    x = make_feature_frame(train, "engineered")
    y = np.log1p(train[TARGET].astype(float).to_numpy())
    kf = KFold(n_splits=folds, shuffle=True, random_state=random_state)
    scores = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(x), start=1):
        estimator = Pipeline(
            [
                ("preprocess", make_preprocessor(x.iloc[tr_idx])),
                (
                    "model",
                    TorchMLPRegressor(
                        max_epochs=90 if quick else 280,
                        patience=14 if quick else 28,
                        random_state=random_state + fold,
                    ),
                ),
            ]
        )
        estimator.fit(x.iloc[tr_idx], y[tr_idx])
        pred = estimator.predict(x.iloc[va_idx])
        score = kaggle_log_rmse(y[va_idx], pred)
        print(f"MLP fold {fold}/{folds}: log-RMSE={score:.6f}")
        scores.append(score)

    return {
        "experiment": "engineered_mlp",
        "feature_profile": "engineered",
        "model": "pytorch_mlp",
        "remove_outliers": False,
        "rows_used": int(len(train)),
        "rows_removed": 0,
        "cv_folds": int(folds),
        "log_rmse_mean": float(np.mean(scores)),
        "log_rmse_std": float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
        "fold_scores": [float(s) for s in scores],
    }





## Chạy notebook


In [ ]:
log("[5] Training MLP...")
train, _ = load_competition_data(PROJECT_ROOT / "data")
result = mlp_cv(train, folds=3, random_state=42, quick=True)
pd.DataFrame([result]).to_csv(OUTPUT / "mlp_results.csv", index=False)
log(f"MLP CV log-RMSE={result['log_rmse_mean']:.6f}")
